# DG-1 walkthrough — closed-form K and pure-dephasing K(t)

**Decision Gate:** DG-1 (Formula implementation; CL-2026-005 v0.4 Entries 1, 3, 4).  
**Verdict:** PASS (2026-04-30).  
**Cards exercised:** [A1](../benchmarks/benchmark_cards/A1_closed-form-K_v0.1.1.yaml), [A3](../benchmarks/benchmark_cards/A3_pure-dephasing_v0.1.1.yaml).  
**Authoritative status:** [`docs/validity_envelope.md`](../docs/validity_envelope.md).

This notebook demonstrates two of the three DG-1 cards end-to-end:

1. **Card A1** (algebraic): the closed-form Letter Eq. (6) operator $K = (1/2id)\sum_\alpha [F_\alpha^\dagger,\, L[F_\alpha]]$ applied to a unitary generator $L[X]=-i[H,X]$. Expectation: $K=H$ (traceless part) at machine precision.
2. **Card A3** (dynamical): the time-dependent effective Hamiltonian $K(t)$ for pure dephasing (system $\sigma_z$ coupled to an Ohmic thermal bath). Expectation: $K(t)=(\omega_r(t)/2)\,\sigma_z$ with $\omega_r(t)\to\omega$ as $t\to\infty$ (Fock-diagonal initial bath).

> **Coordinate-choice note (Sail v0.5 §4):** all $K$ values below are Hayden–Sorce minimal-dissipation gauge-fixed. See [`docs/do_not_cite_as.md`](../docs/do_not_cite_as.md) for citation discipline.

## 0. Setup

In [ ]:
import numpy as np

import cbg
from cbg.basis import matrix_unit_basis, su_d_generator_basis
from cbg.effective_hamiltonian import K_from_generator
from cbg import tcl_recursion as tr

print(f"cbg.__version__       = {cbg.__version__}")
print(f"cbg.__sail_version__  = {cbg.__sail_version__}")
print(f"cbg.__ledger_anchor__ = {cbg.__ledger_anchor__}")

## 1. Card A1 — closed-form $K$ from a unitary generator

For $H_S = (\omega/2)\sigma_z$ and $L[X]=-i[H_S,X]$ the Letter Eq. (6) construction must return $K=H_S$ exactly (traceless $H_S$; gauge trace-removal vanishes).

In [ ]:
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)
omega = 1.0
H_S = 0.5 * omega * sigma_z

def L_unitary(X):
    return -1j * (H_S @ X - X @ H_S)

basis_mu = matrix_unit_basis(d=2)
K = K_from_generator(L_unitary, basis_mu)

print("K =\n", K)
print(f"||K - H_S||_F = {np.linalg.norm(K - H_S):.2e}")

**Basis-independence cross-check** (DG-2 territory, but free here): compute $K$ in the $\mathfrak{su}(d)$-generator basis (normalized Pauli) and verify it matches the matrix-unit value.

In [ ]:
basis_su = su_d_generator_basis(d=2)
K_su = K_from_generator(L_unitary, basis_su)
print(f"||K_matrix-unit - K_su(d)||_F = {np.linalg.norm(K - K_su):.2e}")

## 2. Card A3 — pure-dephasing $K(t)$ on a time grid

Spec (mirrors `A3_pure-dephasing_v0.1.1.yaml`): system $H_S=(\omega/2)\sigma_z$, coupling $A=\sigma_z$, Ohmic spectral density with $\alpha=0.05$, $\omega_c=10\omega$, thermal bath at $k_B T/\hbar\omega = 0.5$, perturbative order $N_{\rm card}=2$.

In [ ]:
t_grid = np.linspace(0.0, 5.0, 41)  # in units of 1/omega

K_t = tr.K_total_thermal_on_grid(
    N_card=2,
    t_grid=t_grid,
    system_hamiltonian=H_S,
    coupling_operator=sigma_z,
    bath_state={"family": "thermal", "temperature": 0.5},
    spectral_density={
        "family": "ohmic",
        "coupling_strength": 0.05,
        "cutoff_frequency": 10.0,
    },
)

print(f"K_t.shape = {K_t.shape}  # (n_t, d, d)")
print(f"K(t=0) =\n{K_t[0]}")
print(f"K(t=t_end) diagonal = {K_t[-1].diagonal().real}")

**Structural check.** The expected form is $K(t)=(\omega_r(t)/2)\sigma_z$, so the off-diagonal entries should vanish, the diagonal should be antisymmetric, and the renormalised frequency is twice the (1,1) entry.

In [ ]:
off_diag_max = max(
    np.max(np.abs(K_t[:, 0, 1])),
    np.max(np.abs(K_t[:, 1, 0])),
)
antisym_residual = np.max(np.abs(K_t[:, 0, 0] + K_t[:, 1, 1]))
omega_r = 2.0 * K_t[:, 0, 0].real

print(f"max |K_off-diagonal|        = {off_diag_max:.2e}")
print(f"max |K_00 + K_11| (antisym) = {antisym_residual:.2e}")
print(f"omega_r(t) sample (t = 0, end-2, end): "
      f"{omega_r[0]:.4f}, {omega_r[-2]:.4f}, {omega_r[-1]:.4f}")

## 3. Outcome

Both expected structural properties hold at machine precision:

- Card A1 returns $K=H_S$ exactly; the algebraic Letter Eq. (6) construction is basis-independent at $d=2$.
- Card A3 returns $K(t)$ proportional to $\sigma_z$ at every grid point, consistent with the analytical Łuczka 1990 / Doll 2008 / Leggett 1987 expressions.

These two cards (plus A4, the $\sigma_x$ thermal sibling) are the DG-1 PASS evidence. See [`benchmarks/results/A3_pure-dephasing_v0.1.1_result.json`](../benchmarks/results/A3_pure-dephasing_v0.1.1_result.json) for the frozen verdict numbers.